# Stochastic Optimization

## 1.1 Random Search

<a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" align="left" src="https://i.creativecommons.org/l/by-sa/4.0/80x15.png" /></a>&nbsp;| Dennis G. Wilson | <a href="https://supaerodatascience.github.io/stochastic/">https://supaerodatascience.github.io/stochastic/</a>

Le principe fondamental de tous les algorithmes d'optimisation stochastique réside dans l'échantillonnage aléatoire de l'espace de recherche.   
  
Notre premier algorithme consistera simplement en ceci : échantillonner aléatoirement l'espace de recherche, point par point, en suivant toujours la même distribution.

Nous commencerons par définir un espace de recherche à l'aide d'ne fonction de test classique pour l'optimisation continue: la fonction de Rosenbrock. Sa forme bidimensionnelle est la suivante:  

$f(x, y)=(a-x)^2+B(y-x^2)^2$

In [ ]:
def rosenbrock(x, y, a=1, B=100):
      return (a-x)**2 + B*((y-x**2))**2

Cette fonction est également disponible dans `scipy` sous sa forme générale, adaptable à $N$ dimensions.

In [ ]:
from scipy.optimize import rosen

print(rosenbrock(0.1, 0.2))
print(rosen([0.1, 0.2]))

Pour comprendre cet espace de recherche, nous allons le représenter graphiquement.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from mpl_toolkits.mplot3d import Axes3D
import math

In [ ]:
def plot_2d(X, Y, Z):
    fig = plt.figure(figsize=(8, 6))
    cs = plt.contourf(X, Y, Z, cmap=cm.nipy_spectral)
    fig.colorbar(cs)
    plt.show()

def plot_3d(X, Y, Z):
    fig = plt.figure(figsize=(6, 4))
    ax = fig.add_subplot(projection='3d')
    surf = ax.plot_surface(X, Y, Z, cmap=cm.nipy_spectral,
                       linewidth=0, antialiased=False)
    fig.colorbar(surf)
    plt.show()

In [ ]:
X = np.arange(-2, 2, 0.01)
Y = np.arange(-2, 2, 0.01)
X, Y = np.meshgrid(X, Y)
Z = rosenbrock(X, Y)
plot_3d(X, Y, Z)

Comme on peut le constater, il existe une zone de vallée correspondant à un minimum. Même en utilisant une recherche non informée, cette vallée devrait être facile à localiser.  
  
Toutefois, le minimum global se situe en $(a, a^2)$, où $f(a, a^2) = 0$, et ce point précis est très difficile à trouver.   
  
Dans notre cas, ce minimum global se trouve en $(1, 1)$. Essayons de le trouver.

Nous allons d'abord effectuer une recherche aléatoire pure, qui utilise une distribution uniforme pour échantillonner l'espace de recherche. Formellement, nous supposerons que:

$f:[A, B]^n \rightarrow \mathbb{R}$

Pour la fonction de Rosenbrock, ce n'est pas tout à fait vrai : la fonction est définie sur l'ensemble des nombres réels $\mathbb{R}$, il n'existe donc pas de valeurs minimale ou maximale pour notre distribution uniforme (en dehors de l'infini). Nous supposerons que nous optimisons uniquement dans l'intervalle [-2, 2] pour les deux dimensions; c'est manifestement tricher, mais cela ne pose pas de problème.

## Pure Random Search

    Initialize uniformly at random x ∈ Ω=[A, B]
    while not terminate
        Sample x′ uniformly at random in Ω
        if f(x′) < f(x)
            x = x'
    return x

Références concernant la recherche aléatoire pure (ou recherche aléatoire uniforme):  
+ S.H.  Brooks:  “Discussion  of  random  methods  for  locating  surface  maxima”.   Operations  Research  6  (1958),  pp.244–251.
+ L.A. Rastrigin: “The convergence of the random search method in the extremal control of a many-parameter system”.Automation and Remote Control 24 (1963), pp.  1337–1342.

Écrivons un algorithme de recherche aléatoire pure et appliquons-le à la fonction de Rosenbrock.  
    
Nous allons l'exécutez-le pendant 1 000 évaluations (c'est-à-dire 1 000 appels à la fonction `rosenbrock`).  
    
Vers quelle valeur converge-t-il ? Tracons la courbe de convergence en suivant l'évolution de la meilleure valeur de f(x) au fil du temps. Si nous relançons l'algorithme, les résultats changent-ils ? Si oui, pourquoi ?

In [ ]:
es = []
fs = []
min_f = np.inf
for i in range(1000):
    x = np.random.rand()*4-2
    y = np.random.rand()*4-2
    f = rosenbrock(x, y)
    if f < min_f:
        min_f = f
        es += [i]
        fs += [f]
        print(i,f)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.yscale('log')
plt.plot(es, fs);

Bien que la fonction de Rosenbrock possède un minimum exact difficile à localiser, on constate qu'une recherche aléatoire uniforme permet de descendre rapidement vers la vallée où $f(x, y) \approx 0$.   
  
Il existe d'autres problèmes pour lesquels une recherche aléatoire pure rencontrera beaucoup plus de difficultés.  
  
Vous trouverez une liste de fonctions classiques [ici](https://en.wikipedia.org/wiki/Test_functions_for_optimization).   

<div class="alert alert-success">
    <h3>Exercice 1</h3>  
    
Appliquez une recherche aléatoire à la fonction de Himmelblau. Analysez les résultats.  
</div>

In [ ]:
def himmelblau(x, y):
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

In [ ]:
X = np.arange(-5, 5, 0.01)
Y = np.arange(-5, 5, 0.01)
X, Y = np.meshgrid(X, Y)
Z = himmelblau(X, Y)
plot_3d(X, Y, Z)

In [ ]:
#Exercice 1 - voir RS_Solution1

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.yscale('log')
plt.plot(es, fs);

Effectuer une recherche en utilisant une distribution uniforme pour chaque échantillon ne repose sur aucune information: les meilleures valeurs trouvées par l'algorithme jusqu'alors n'orientent pas l'échantillonnage suivant.  
  
  
Il est évident qu'en exploitant ces échantillons précédents, nous pouvons guider l'algorithme vers les zones à explorer ensuite. C'est là le principe de l'**optimisation aléatoire**.

## Random Optimization

Let $f: \mathbb{R}^n → \mathbb{R}$ be the fitness or cost function which must be minimized. Let $x ∈ \mathbb{R}^n$ designate a position or candidate solution in the search-space.

    Initialize x randomly in ℝ
    while not terminate
        x' = x + N(0, 1)
        if f(x′) < f(x)
            x = x'
    return x

where $N(0,1)$ is a normal distribution centered at 0 with a standard deviation of 1.

<div class="alert alert-success">
    <h3>Exercice 2</h3>

Modifiez l'algorithme de recherche aléatoire pure pour implémenter une optimisation aléatoire utilisant une loi normale afin de déterminer le point suivant.  
  
Appliquez-le à la fonction de Rosenbrock et laissez-le s'exécuter pendant 1000 évaluations.  
  
Cette méthode améliore-t-elle les résultats par rapport à la recherche aléatoire pure?  

Tracez un graphique comparant les deux approches. 

</div>

In [ ]:
#Exercice 2 - voir RS_Solution2

L'algorithme d'optimisation aléatoire correspond exactement à la stratégie d'évolution **(1+1)** que nous aborderons ensuite, et que l'on désigne aussi souvent sous le nom de «recherche par ascension de colline» (*hill climbing*).  
  
Quels aspects de cet algorithme pourraient encore être améliorés ? À quel niveau la recherche manque-t-elle encore d'information ?